# 02 - Classical Baselines (Paper-Quality)

Evaluates classical z-axis interpolation baselines across ALL test cases and multiple sparse ratios.

## Outputs
- Per-case JSON results with PSNR, SSIM, MAE, and ROI metrics
- Aggregated summary CSV for paper tables
- Runtime benchmarks

## Methods
- Nearest neighbor
- Linear interpolation
- Cubic interpolation

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True
SPARSE_RATIOS = [2, 3, 4]
SEED = 42
# ===================================================================

In [ ]:
import os, sys, json, subprocess, time
import numpy as np
from pathlib import Path

WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"
if not (REPO_DIR / "src").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.utils.seed import set_seed
set_seed(SEED)

In [ ]:
# Load prepared data info
prepared_dir = OUTPUT_ROOT / "prepared"
split_path = prepared_dir / "split_info.json"

if split_path.exists():
    with open(split_path) as f:
        split_info = json.load(f)
    test_cases = split_info["test"]
else:
    raise FileNotFoundError("Run notebook 01 first to prepare data.")

if not FULL_EXPERIMENT:
    test_cases = test_cases[:2]
    SPARSE_RATIOS = [2, 3]

print(f"Test cases: {len(test_cases)}")
print(f"Sparse ratios: {SPARSE_RATIOS}")

In [ ]:
from tqdm import tqdm
from src.models.classical_interp import ClassicalInterpolator
from src.evaluation.metrics import evaluate_volume
from src.data.sparse_simulator import SparseSimulator

organ_labels = {"liver": 1, "bladder": 2, "lungs": 3, "kidneys": 4, "bone": 5}

all_results = []

for R in SPARSE_RATIOS:
    out_dir = OUTPUT_ROOT / "classical_baselines" / f"R{R}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"  Sparse Ratio R={R}")
    print(f"{'='*60}")

    for case_idx in tqdm(test_cases, desc=f"R={R}"):
        vol_path = prepared_dir / f"volume_case{case_idx}.npy"
        if not vol_path.exists():
            print(f"  Skipping case {case_idx}: not prepared")
            continue

        volume = np.load(vol_path)
        labels = None
        lab_path = prepared_dir / f"labels_case{case_idx}.npy"
        if lab_path.exists():
            labels = np.load(lab_path)

        sim = SparseSimulator(sparse_ratio=R)
        sparse = sim.simulate(volume)

        # Run all classical methods and time them
        t0 = time.time()
        results = ClassicalInterpolator.interpolate_all_methods(
            observed_slices=sparse["observed_slices"],
            observed_indices=sparse["observed_indices"],
            target_indices=sparse["target_indices"],
        )
        interp_time = time.time() - t0

        gt_targets = sparse["target_slices"]

        case_results = {}
        for method, pred in results.items():
            ev = evaluate_volume(
                predictions=pred,
                ground_truths=gt_targets,
                target_indices=sparse["target_indices"],
                labels=labels,
                organ_labels=organ_labels,
            )
            ev["summary"]["runtime_s"] = interp_time / len(results)
            case_results[method] = ev

            all_results.append({
                "method": method,
                "case_idx": int(case_idx),
                "sparse_ratio": R,
                **ev["summary"],
            })

        # Save per-case results
        with open(out_dir / f"case{case_idx}.json", "w") as f:
            json.dump(case_results, f, indent=2)

print(f"\nTotal evaluations: {len(all_results)}")

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

# Summary table for paper (Table 1)
print("\n=== CLASSICAL BASELINES - Summary Table (Paper Table 1) ===")
summary = df.groupby(["sparse_ratio", "method"]).agg({
    "mean_psnr": ["mean", "std"],
    "mean_ssim": ["mean", "std"],
    "mean_mae": ["mean", "std"],
}).round(4)
print(summary.to_string())

# ROI metrics summary
print("\n=== ROI Metrics (if available) ===")
roi_rows = []
for _, row in df.iterrows():
    if "roi" in row and isinstance(row.get("roi"), dict):
        for organ, vals in row["roi"].items():
            roi_rows.append({
                "method": row["method"],
                "sparse_ratio": row["sparse_ratio"],
                "organ": organ,
                **vals,
            })
if roi_rows:
    df_roi = pd.DataFrame(roi_rows)
    print(df_roi.groupby(["sparse_ratio", "method", "organ"]).mean().round(4).to_string())

# Save aggregated results
df.to_csv(OUTPUT_ROOT / "classical_baselines" / "summary.csv", index=False)
print("\nSaved summary to:", OUTPUT_ROOT / "classical_baselines" / "summary.csv")